In [28]:
import requests
import json
import pandas as pd

print("Requests library imported successfully!")

Requests library imported successfully!


In [17]:
#define api endpoint and parameters

url ="https://api.open-meteo.com/v1/forecast"
print(url)

params= {
    "latitude": 27.7172,
    "longitude": 85.3240,
    "daily":[
        "temperature_2m_max",
        "temperature_2m_min",
    ],
    "timezone" : "Asia/Kathmandu"
}

https://api.open-meteo.com/v1/forecast


In [18]:
#send api get request and error handling

try:
    response = requests.get(url, params=params, timeout = 30)

    response.raise_for_status() #raise an exception for 4xx and 5xx status codes

    weather_data = response.json() #convert the response into Python objects

    print("API extraction successful!")

except requests.exceptions.Timeout:
    print("The API request timed out.")

except requests.exceptions.HTTPError as error:
    print("HTTP error:", error)

except requests.exceptions.RequestException as error:
    print("Error occurred during the API request:", error)

except ValueError as error:
    print("Error occurred while parsing the JSON response:", error)


API extraction successful!


In [19]:
#inspect the response

print("Response status code:", response.status_code)
print("Requested URL:", response.url)

Response status code: 200
Requested URL: https://api.open-meteo.com/v1/forecast?latitude=27.7172&longitude=85.324&daily=temperature_2m_max&daily=temperature_2m_min&timezone=Asia%2FKathmandu


In [20]:
print(type(weather_data)) #converted the JSON reponse body into Python dictionary

<class 'dict'>


In [21]:
print(json.dumps(weather_data, indent=4)) #pretty print the JSON response body

{
    "latitude": 27.732864,
    "longitude": 85.34831,
    "generationtime_ms": 0.044465065002441406,
    "utc_offset_seconds": 20700,
    "timezone": "Asia/Kathmandu",
    "timezone_abbreviation": "GMT+5:45",
    "elevation": 1301.0,
    "daily_units": {
        "time": "iso8601",
        "temperature_2m_max": "\u00b0C",
        "temperature_2m_min": "\u00b0C"
    },
    "daily": {
        "time": [
            "2026-10-31",
            "2026-11-01",
            "2026-11-02",
            "2026-11-03",
            "2026-11-04",
            "2026-11-05",
            "2026-11-06"
        ],
        "temperature_2m_max": [
            18.7,
            24.0,
            23.3,
            23.2,
            23.8,
            23.1,
            23.6
        ],
        "temperature_2m_min": [
            14.2,
            13.7,
            14.0,
            12.3,
            10.6,
            9.7,
            8.4
        ]
    }
}


In [26]:
#Access the daily dictionary

daily_data = weather_data["daily"]
print(type(daily_data)) 
print(daily_data.keys())

#acessing indivuduals lists

dates = daily_data["time"]
max_temps = daily_data["temperature_2m_max"]
min_temps = daily_data["temperature_2m_min"]

print("Dates:", dates)
print("Max Temperatures:", max_temps)
print("Min Temperatures:", min_temps)
                       

<class 'dict'>
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min'])
Dates: ['2026-10-31', '2026-11-01', '2026-11-02', '2026-11-03', '2026-11-04', '2026-11-05', '2026-11-06']
Max Temperatures: [18.7, 24.0, 23.3, 23.2, 23.8, 23.1, 23.6]
Min Temperatures: [14.2, 13.7, 14.0, 12.3, 10.6, 9.7, 8.4]


In [27]:
#inspecting the lengths

print("Length of Dates:", len(dates))
print("Length of Max Temperatures:", len(max_temps))
print("Length of Min Temperatures:", len(min_temps))

Length of Dates: 7
Length of Max Temperatures: 7
Length of Min Temperatures: 7


In [29]:
#Extracted the data into a structured format

weather_df = pd.DataFrame({
    "Date": dates,  
    "temperature_2m_max": max_temps,
    "temperature_2m_min": min_temps
})

weather_df

,Date,temperature_2m_max,temperature_2m_min
0,2026-10-31,18.7,14.2
1,2026-11-01,24.0,13.7
2,2026-11-02,23.3,14.0
3,2026-11-03,23.2,12.3
4,2026-11-04,23.8,10.6
5,2026-11-05,23.1,9.7
6,2026-11-06,23.6,8.4


In [35]:
#validating the data

print("Shape of the dataset:", weather_df.shape)

print("Column names:", weather_df.columns)
print("Datatypes: ", weather_df.dtypes)
print("Dataset information:", weather_df.info())

Shape of the dataset: (7, 3)
Column names: Index(['Date', 'temperature_2m_max', 'temperature_2m_min'], dtype='str')
Datatypes:  Date                      str
temperature_2m_max    float64
temperature_2m_min    float64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                7 non-null      str    
 1   temperature_2m_max  7 non-null      float64
 2   temperature_2m_min  7 non-null      float64
dtypes: float64(2), str(1)
memory usage: 300.0 bytes
Dataset information: None


In [38]:
#convert the date into datatime format

weather_df["Date"]= pd.to_datetime(weather_df["Date"])
print(weather_df.dtypes)

Date                  datetime64[us]
temperature_2m_max           float64
temperature_2m_min           float64
dtype: object


In [40]:
#checking for missing values

print(weather_df.isnull().sum())

Date                  0
temperature_2m_max    0
temperature_2m_min    0
dtype: int64


In [42]:
#checking  the temperature consistency

invalid_records = weather_df[ 
    weather_df["temperature_2m_max"] < weather_df["temperature_2m_min"]
    ]

print("Invalid records:",invalid_records)

Invalid records: Empty DataFrame
Columns: [Date, temperature_2m_max, temperature_2m_min]
Index: []


In [43]:
#Transforming the dataset

#creating a derived column

weather_df["temperature_range"] = weather_df["temperature_2m_max"] - weather_df["temperature_2m_min"]
weather_df

                                            

,Date,temperature_2m_max,temperature_2m_min,temperature_range
0,2026-10-31,18.7,14.2,4.5
1,2026-11-01,24.0,13.7,10.3
2,2026-11-02,23.3,14.0,9.3
3,2026-11-03,23.2,12.3,10.9
4,2026-11-04,23.8,10.6,13.2
5,2026-11-05,23.1,9.7,13.4
6,2026-11-06,23.6,8.4,15.2


In [53]:
# #saving the transformed dataset to a CSV file

from  pathlib import Path

data_dir = Path.cwd().parent / "data"
output_path = data_dir / "kathmandu_weather_data.csv"

weather_df.to_csv(
    output_path,
    index=False #tells not to write index columns
)
print("Saved to:", output_path)
print("File exists:", output_path.exists())

Saved to: g:\weather-data-pipeline\data\kathmandu_weather_data.csv
File exists: True


Making Reusable Python Functions

In [54]:
def calculate_temperature_range(max_temp, min_temp):

    return max_temp - min_temp

In [55]:
result = calculate_temperature_range(30, 20)
print(result)

10


In [59]:
#applying it to pandas dataframe

weather_df["temperature_range"] = weather_df.apply(
    lambda row : calculate_temperature_range(
        row["temperature_2m_max"],
        row["temperature_2m_min"]
    ),
    axis = 1
)